In [0]:
%run ../00_common/data_utils

In [0]:
def calc_consumer_crossbrand_optin_master(task_id):
    """
    计算并更新consumer crossbrand optin主数据
    所有Region逻辑一致，只检查optin_flag变化
    
    Args:
        task_id: 任务ID
    """
    master_crossbrand_optin_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_crossbrand_optin"
    
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'") \
        .filter(F.col("IS_INCLUDE") == True) 

    itermediate_crossbrand_optin_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_crossbrand_optin") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_crossbrand_optin_df = spark.table(master_crossbrand_optin_table_name)
    
    # 2. 查询query_crossbrandoptin (步骤4提前)
    master_crossbrand_optin_to_insert = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_crossbrand_optin_df.alias("srbo"),
            (F.col("srcc.SRCC_ID") == F.col("srbo.SRBO_SRCC_ID")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srbo.srbo_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srbo.srbo_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srbo.srbo_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_crossbrand_optin_df.alias("scbo"),
            (F.col("scon.scon_id") == F.col("scbo.scbo_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scbo.scbo_mrkt_code")),
            "left"
        )
        .where(
            F.coalesce(F.col("srbo.srbo_optin_flag").cast("String"), F.lit("x")) != 
            F.coalesce(F.col("scbo.scbo_optin_flag").cast("String"), F.lit("x"))
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srbo.srbo_mrkt_code"),
            # 动态optin_dt: DELETE时使用srcc_sourcetimestamp
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.col("srcc.srcc_sourcetimestamp")
            ).otherwise(F.col("srbo.srbo_optin_dt")).alias("srbo_optin_dt"),
            # 动态optin_flag: DELETE时设为0
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit(False)
            ).otherwise(F.col("srbo.srbo_optin_flag")).alias("srbo_optin_flag"),
            F.col("srbo.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scbo_id"),
            F.col("scon_id").alias("scbo_scon_id"),
            F.col("srbo_mrkt_code").alias("scbo_mrkt_code"),
            F.col("srbo_optin_dt").alias("scbo_optin_dt"),
            F.col("srbo_optin_flag").alias("scbo_optin_flag"),
            F.current_timestamp().alias("scbo_creation_dt"),
            F.lit("ELC").alias("scbo_creation_uid"),
            F.current_timestamp().alias("scbo_update_dt"),
            F.lit("ELC").alias("scbo_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_crossbrand_optin_to_insert = master_crossbrand_optin_to_insert.checkpoint(eager=True)
    master_crossbrand_optin_insert_count = master_crossbrand_optin_to_insert.count()
    
    # 3. 查询query_crossbrandoptin_delete
    master_crossbrand_optin_to_delete = (
        itermediate_crossbrand_optin_df.alias("srbo")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srbo.srbo_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srbo.srbo_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_crossbrand_optin_df.alias("scbo"),
            (F.col("scon.scon_id") == F.col("scbo.scbo_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scbo.scbo_mrkt_code")),
            "inner"
        )
        .where(
            F.coalesce(F.col("srbo.srbo_optin_flag").cast("String"), F.lit("x")) != 
            F.coalesce(F.col("scbo.scbo_optin_flag").cast("String"), F.lit("x"))
        )
        .select(F.col("scbo.*")).distinct()
    )

    # 4. 执行删除操作
    master_crossbrand_optin_delete_count = 0
    if not master_crossbrand_optin_to_delete.isEmpty():
        master_crossbrand_optin_delta_table = DeltaTable.forName(spark, master_crossbrand_optin_table_name)
        master_crossbrand_optin_merge_result = (
            master_crossbrand_optin_delta_table.alias("target")
            .merge(
                master_crossbrand_optin_to_delete.alias("source"),
                """
                target.scbo_id = source.scbo_id AND
                target.scbo_mrkt_code = source.scbo_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_crossbrand_optin_delete_count = max(master_crossbrand_optin_to_delete.count(), 0)

    # 5. 插入数据到目标表
    if master_crossbrand_optin_insert_count > 0:
        append_table(master_crossbrand_optin_to_insert, master_crossbrand_optin_table_name)
    print(f"master crossbrand optin inserted count: {master_crossbrand_optin_insert_count}, deleted count: {master_crossbrand_optin_delete_count}")

In [0]:
def calc_consumer_auxiliaryattribute_master(task_id):
    """
    计算并更新consumer auxiliaryattribute主数据
    全量替换模式：删除特定scon_id的所有auxiliaryattribute记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_auxiliaryattribute_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_auxiliary_attribute"
    
    # 1. 初始化数据源
    itermediate_auxiliaryattribute_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_auxiliary_attribute") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_auxiliaryattribute_df = spark.table(master_auxiliaryattribute_table_name)
    
    # 2. 查询要插入的auxiliaryattribute数据（步骤提前）
    master_auxiliaryattribute_to_insert = (
        itermediate_auxiliaryattribute_df.alias("sraa")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("sraa.sraa_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("sraa.sraa_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("sraa.sraa_mrkt_code"),
            F.col("sraa.SRAA_CODE"),
            F.col("sraa.SRAA_DESC"),
            F.col("sraa.SRAA_MULTIVALUEFLAG"),
            F.col("sraa.SRAA_VALUE"),
            F.col("sraa.SRAA_ACTIVE_FLAG"),
            F.col("sraa.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scaa_id"),
            F.col("scon_id").alias("scaa_scon_id"),
            F.col("sraa_mrkt_code").alias("scaa_mrkt_code"),
            F.col("SRAA_CODE").alias("scaa_code"),
            F.col("SRAA_DESC").alias("scaa_desc"),
            F.col("SRAA_MULTIVALUEFLAG").alias("scaa_multivalueflag"),
            F.col("SRAA_VALUE").alias("scaa_value"),
            F.col("SRAA_ACTIVE_FLAG").alias("scaa_active_flag"),
            F.current_timestamp().alias("scaa_creation_dt"),
            F.lit("ELC").alias("scaa_creation_uid"),
            F.current_timestamp().alias("scaa_update_dt"),
            F.lit("ELC").alias("scaa_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_auxiliaryattribute_to_insert = master_auxiliaryattribute_to_insert.checkpoint(eager=True)
    master_auxiliaryattribute_insert_count = master_auxiliaryattribute_to_insert.count()
    
    # 3. 查询要删除的auxiliaryattribute数据
    scon_ids_to_delete = (
        itermediate_auxiliaryattribute_df.alias("sraa")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("sraa.sraa_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("sraa.SRAA_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_auxiliaryattribute_to_delete = (
        master_auxiliaryattribute_df.alias("scaa")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scaa.scaa_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scaa.scaa_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scaa.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_auxiliaryattribute_delete_count = 0
    if not master_auxiliaryattribute_to_delete.isEmpty():
        master_auxiliaryattribute_delta_table = DeltaTable.forName(spark, master_auxiliaryattribute_table_name)
        master_auxiliaryattribute_merge_result = (
            master_auxiliaryattribute_delta_table.alias("target")
            .merge(
                master_auxiliaryattribute_to_delete.alias("source"),
                """
                target.scaa_id = source.scaa_id AND
                target.scaa_mrkt_code = source.scaa_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_auxiliaryattribute_delete_count = max(master_auxiliaryattribute_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_auxiliaryattribute_insert_count > 0:
        append_table(master_auxiliaryattribute_to_insert, master_auxiliaryattribute_table_name)
    
    print(f"master auxiliaryattribute inserted count: {master_auxiliaryattribute_insert_count}, deleted count: {master_auxiliaryattribute_delete_count}")

In [0]:
def calc_consumer_hairconcerns_master(task_id):
    """
    计算并更新consumer hairconcerns主数据
    全量替换模式：删除特定scon_id的所有hairconcerns记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_hairconcerns_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_hair_concerns"
    
    # 1. 初始化数据源
    itermediate_hairconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hair_concerns") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_hairconcerns_df = spark.table(master_hairconcerns_table_name)
    
    # 2. 查询要插入的hairconcerns数据（步骤提前）
    master_hairconcerns_to_insert = (
        itermediate_hairconcerns_df.alias("srhc")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srhc.srhc_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srhc.srhc_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srhc.srhc_mrkt_code"),
            F.col("srhc.SRHC_CONCERN_DESC").alias("Hair_Concern"),
            F.col("srhc.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("schc_id"),
            F.col("scon_id").alias("schc_scon_id"),
            F.col("srhc_mrkt_code").alias("schc_mrkt_code"),
            F.col("Hair_Concern").alias("schc_hairconcern_desc"),
            F.current_timestamp().alias("schc_creation_dt"),
            F.lit("ELC").alias("schc_creation_uid"),
            F.current_timestamp().alias("schc_update_dt"),
            F.lit("ELC").alias("schc_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_hairconcerns_to_insert = master_hairconcerns_to_insert.checkpoint(eager=True)
    master_hairconcerns_insert_count = master_hairconcerns_to_insert.count()
    
    # 3. 查询要删除的hairconcerns数据
    scon_ids_to_delete = (
        itermediate_hairconcerns_df.alias("srhc")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srhc.srhc_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srhc.SRHC_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    master_hairconcerns_to_delete = (
        master_hairconcerns_df.alias("schc")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("schc.schc_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("schc.schc_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("schc.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_hairconcerns_delete_count = 0
    if not master_hairconcerns_to_delete.isEmpty():
        master_hairconcerns_delta_table = DeltaTable.forName(spark, master_hairconcerns_table_name)
        master_hairconcerns_merge_result = (
            master_hairconcerns_delta_table.alias("target")
            .merge(
                master_hairconcerns_to_delete.alias("source"),
                """
                target.schc_id = source.schc_id AND
                target.schc_mrkt_code = source.schc_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_hairconcerns_delete_count = max(master_hairconcerns_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_hairconcerns_insert_count > 0:
        append_table(master_hairconcerns_to_insert, master_hairconcerns_table_name)
    
    print(f"master hairconcerns inserted count: {master_hairconcerns_insert_count}, deleted count: {master_hairconcerns_delete_count}")

In [0]:
def calc_consumer_makeupconcerns_master(task_id):
    """
    计算并更新consumer makeupconcerns主数据
    全量替换模式：删除特定scon_id的所有makeupconcerns记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_makeupconcerns_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_makeup_concerns"
    
    # 1. 初始化数据源
    itermediate_makeupconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_makeup_concerns") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_makeupconcerns_df = spark.table(master_makeupconcerns_table_name)
    
    # 2. 查询要插入的makeupconcerns数据（步骤提前）
    master_makeupconcerns_to_insert = (
        itermediate_makeupconcerns_df.alias("srmc")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srmc.srmc_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srmc.srmc_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srmc.srmc_mrkt_code"),
            F.col("srmc.SRMC_CONCERN_DESC").alias("Makeup_Concern"),
            F.col("srmc.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scmc_id"),
            F.col("scon_id").alias("scmc_scon_id"),
            F.col("srmc_mrkt_code").alias("scmc_mrkt_code"),
            F.col("Makeup_Concern").alias("scmc_makeupconcern_desc"),
            F.current_timestamp().alias("scmc_creation_dt"),
            F.lit("ELC").alias("scmc_creation_uid"),
            F.current_timestamp().alias("scmc_update_dt"),
            F.lit("ELC").alias("scmc_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_makeupconcerns_to_insert = master_makeupconcerns_to_insert.checkpoint(eager=True)
    master_makeupconcerns_insert_count = master_makeupconcerns_to_insert.count()
    
    # 3. 查询要删除的makeupconcerns数据
    scon_ids_to_delete = (
        itermediate_makeupconcerns_df.alias("srmc")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srmc.srmc_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srmc.SRMC_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    master_makeupconcerns_to_delete = (
        master_makeupconcerns_df.alias("scmc")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scmc.scmc_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scmc.scmc_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scmc.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_makeupconcerns_delete_count = 0
    if not master_makeupconcerns_to_delete.isEmpty():
        master_makeupconcerns_delta_table = DeltaTable.forName(spark, master_makeupconcerns_table_name)
        master_makeupconcerns_merge_result = (
            master_makeupconcerns_delta_table.alias("target")
            .merge(
                master_makeupconcerns_to_delete.alias("source"),
                """
                target.scmc_id = source.scmc_id AND
                target.scmc_mrkt_code = source.scmc_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_makeupconcerns_delete_count = max(master_makeupconcerns_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_makeupconcerns_insert_count > 0:
        append_table(master_makeupconcerns_to_insert, master_makeupconcerns_table_name)
    
    print(f"master makeupconcerns inserted count: {master_makeupconcerns_insert_count}, deleted count: {master_makeupconcerns_delete_count}")

In [0]:
def calc_consumer_beautyconcerns_master(task_id):
    # 1. 初始化表名
    master_beautyconcerns_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_beauty_concerns"
    # 2. 初始化数据源
    itermediate_hairconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hair_concerns") \
        .where(f"TASK_ID = '{task_id}'") 

    itermediate_makeupconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_makeup_concerns") \
        .where(f"TASK_ID = '{task_id}'") 

    itermediate_skinconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_skin_concerns") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_beautyconcerns_df = spark.table(master_beautyconcerns_table_name)
    
    # 3. 查询要删除的beautyconcerns数据 (对应query_beautyconcerns_delete)
    # 注意：这里使用了hairconcerns表，与SQL一致
    scon_ids_to_delete = (
        itermediate_hairconcerns_df.alias("srhc")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_srcc_id") == F.col("srhc.SRHC_SRCC_ID")) &
            (F.col("sc.scon_mrkt_code") == F.col("srhc.SRHC_MRKT_CODE")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    # 获取要删除的完整beautyconcerns记录
    master_beautyconcerns_to_delete = (
        master_beautyconcerns_df.alias("scbc")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scbc.scbc_mrkt_code") == F.col("del_ids.scon_mrkt_code")) &
            (F.col("scbc.scbc_scon_id") == F.col("del_ids.scon_id")),
            "inner"
        )
        .select(F.col("scbc.*")).distinct()
    )
    
    # 4. DeltaTable merge删除操作
    master_beautyconcerns_delete_count = 0
    if not master_beautyconcerns_to_delete.isEmpty():
        master_beautyconcerns_delta_table = DeltaTable.forName(spark, master_beautyconcerns_table_name)
        master_beautyconcerns_merge_result = (
            master_beautyconcerns_delta_table.alias("target")
            .merge(
                master_beautyconcerns_to_delete.alias("source"),
                """
                target.scbc_id = source.scbc_id AND
                target.scbc_mrkt_code = source.scbc_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        # master_beautyconcerns_delete_count = master_beautyconcerns_merge_result.first().num_deleted_rows
    
    # 5. 查询要插入的beautyconcerns数据 (对应query_beautyconcerns)
    master_beautyconcerns_to_insert = (
        itermediate_hairconcerns_df.alias("srhc")
        .join(
            itermediate_makeupconcerns_df.alias("srmc"),
            (F.col("srhc.srhc_srcc_id") == F.col("srmc.srmc_srcc_id")) &
            (F.col("srhc.srhc_mrkt_code") == F.col("srmc.srmc_mrkt_code")),
            "inner"
        )
        .join(
            itermediate_skinconcerns_df.alias("srsk"),
            (F.col("srmc.srmc_srcc_id") == F.col("srsk.srsk_srcc_id")) &
            (F.col("srmc.srmc_mrkt_code") == F.col("srsk.srsk_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srsk.srsk_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srsk.srsk_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srhc.srhc_mrkt_code"),
            F.col("srhc.SRHC_CONCERN_DESC").alias("Hair_Concern"),
            F.col("srmc.SRMC_CONCERN_DESC").alias("Makeup_Concern"),
            F.col("srsk.SRSK_CONCERN_DESC").alias("Skin_Concern"),
            F.col("srhc.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scbc_id"),
            F.col("scon_id").alias("scbc_scon_id"),
            F.col("srhc_mrkt_code").alias("scbc_mrkt_code"),
            F.col("Hair_Concern").alias("scbc_hair_concern"),
            F.col("Makeup_Concern").alias("scbc_makeup_concern"),
            F.col("Skin_Concern").alias("scbc_skin_concern"),
            F.current_timestamp().alias("scbc_creation_dt"),
            F.lit("ELC").alias("scbc_creation_uid"),
            F.current_timestamp().alias("scbc_update_dt"),
            F.lit("ELC").alias("scbc_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_beautyconcerns_insert_count = max(master_beautyconcerns_to_insert.count(), 0)
    # 6. 插入数据到目标表
    if master_beautyconcerns_insert_count > 0:
        append_table(master_beautyconcerns_to_insert, master_beautyconcerns_table_name)
    print(f"master beautyconcerns inserted count: {master_beautyconcerns_insert_count}, deleted count: {master_beautyconcerns_delete_count}")

In [0]:
def calc_consumer_skinconcerns_master(task_id):
    """
    计算并更新consumer skinconcerns主数据
    全量替换模式：删除特定scon_id的所有skinconcerns记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_skinconcerns_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_skin_concerns"
    
    # 1. 初始化数据源
    itermediate_skinconcerns_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_skin_concerns") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_skinconcerns_df = spark.table(master_skinconcerns_table_name)
    
    # 2. 查询要插入的skinconcerns数据（步骤提前）
    master_skinconcerns_to_insert = (
        itermediate_skinconcerns_df.alias("srsk")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srsk.srsk_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srsk.srsk_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srsk.srsk_mrkt_code"),
            F.col("srsk.SRSK_CONCERN_DESC").alias("Skin_Concern"),
            F.col("srsk.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scsc_id"),
            F.col("scon_id").alias("scsc_scon_id"),
            F.col("srsk_mrkt_code").alias("scsc_mrkt_code"),
            F.col("Skin_Concern").alias("scsc_skinconcern_desc"),
            F.current_timestamp().alias("scsc_creation_dt"),
            F.lit("ELC").alias("scsc_creation_uid"),
            F.current_timestamp().alias("scsc_update_dt"),
            F.lit("ELC").alias("scsc_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_skinconcerns_to_insert = master_skinconcerns_to_insert.checkpoint(eager=True)
    master_skinconcerns_insert_count = master_skinconcerns_to_insert.count()
    
    # 3. 查询要删除的skinconcerns数据
    scon_ids_to_delete = (
        itermediate_skinconcerns_df.alias("srsk")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srsk.srsk_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srsk.SRSK_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_skinconcerns_to_delete = (
        master_skinconcerns_df.alias("scsc")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scsc.scsc_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scsc.scsc_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scsc.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_skinconcerns_delete_count = 0
    if not master_skinconcerns_to_delete.isEmpty():
        master_skinconcerns_delta_table = DeltaTable.forName(spark, master_skinconcerns_table_name)
        master_skinconcerns_merge_result = (
            master_skinconcerns_delta_table.alias("target")
            .merge(
                master_skinconcerns_to_delete.alias("source"),
                """
                target.scsc_id = source.scsc_id AND
                target.scsc_mrkt_code = source.scsc_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_skinconcerns_delete_count = max(master_skinconcerns_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_skinconcerns_insert_count > 0:
        append_table(master_skinconcerns_to_insert, master_skinconcerns_table_name)
    
    print(f"master skinconcerns inserted count: {master_skinconcerns_insert_count}, deleted count: {master_skinconcerns_delete_count}")

In [0]:
def calc_consumer_customattr_master(task_id):
    """
    计算并更新consumer customattr主数据
    全量替换模式：删除特定scon_id的所有customattr记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_customattr_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_custom_attributes"
    
    # 1. 初始化数据源
    itermediate_customattr_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_custom_attributes") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_customattr_df = spark.table(master_customattr_table_name)
    
    # 2. 查询要插入的customattr数据（步骤提前）
    master_customattr_to_insert = (
        itermediate_customattr_df.alias("srat")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srat.srat_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srat.srat_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srat.srat_mrkt_code"),
            F.col("srat.SRAT_NAME"),
            F.col("srat.SRAT_VALUE"),
            F.col("srat.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("sccu_id"),
            F.col("scon_id").alias("sccu_scon_id"),
            F.col("srat_mrkt_code").alias("sccu_mrkt_code"),
            F.col("SRAT_NAME").alias("sccu_name"),
            F.col("SRAT_VALUE").alias("sccu_value"),
            F.current_timestamp().alias("sccu_creation_dt"),
            F.lit("ELC").alias("sccu_creation_uid"),
            F.current_timestamp().alias("sccu_update_dt"),
            F.lit("ELC").alias("sccu_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_customattr_to_insert = master_customattr_to_insert.checkpoint(eager=True)
    master_customattr_insert_count = master_customattr_to_insert.count()
    
    # 3. 查询要删除的customattr数据
    scon_ids_to_delete = (
        itermediate_customattr_df.alias("srat")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srat.srat_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srat.SRAT_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_customattr_to_delete = (
        master_customattr_df.alias("sccu")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("sccu.sccu_mrkt_code") == F.col("del_ids.scon_mrkt_code")) &
            (F.col("sccu.sccu_scon_id") == F.col("del_ids.scon_id")),
            "inner"
        )
        .select(F.col("sccu.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_customattr_delete_count = 0
    if not master_customattr_to_delete.isEmpty():
        master_customattr_delta_table = DeltaTable.forName(spark, master_customattr_table_name)
        master_customattr_merge_result = (
            master_customattr_delta_table.alias("target")
            .merge(
                master_customattr_to_delete.alias("source"),
                """
                target.sccu_id = source.sccu_id AND
                target.sccu_mrkt_code = source.sccu_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_customattr_delete_count = max(master_customattr_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_customattr_insert_count > 0:
        append_table(master_customattr_to_insert, master_customattr_table_name)
    
    print(f"master customattr inserted count: {master_customattr_insert_count}, deleted count: {master_customattr_delete_count}")

In [0]:
def calc_consumer_group_master(task_id):
    """
    计算并更新consumer group主数据
    全量替换模式：删除特定scon_id的所有consumergroup记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_consumergroup_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer_group"
    
    # 1. 初始化数据源
    itermediate_consumergroup_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumergroup") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_consumergroup_df = spark.table(master_consumergroup_table_name)
    
    # 2. 查询要插入的consumergroup数据（步骤提前）
    master_consumergroup_to_insert = (
        itermediate_consumergroup_df.alias("srcg")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srcg.srcg_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srcg.srcg_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srcg.srcg_mrkt_code"),
            F.col("srcg.srcg_consumer_grp"),
            F.col("srcg.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scgr_id"),
            F.col("scon_id").alias("scgr_scon_id"),
            F.col("srcg_mrkt_code").alias("scgr_mrkt_code"),
            F.col("srcg_consumer_grp").alias("scgr_consumer_grp"),
            F.current_timestamp().alias("scgr_creation_dt"),
            F.lit("ELC").alias("scgr_creation_uid"),
            F.current_timestamp().alias("scgr_update_dt"),
            F.lit("ELC").alias("scgr_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_consumergroup_to_insert = master_consumergroup_to_insert.checkpoint(eager=True)
    master_consumergroup_insert_count = master_consumergroup_to_insert.count()
    
    # 3. 查询要删除的consumergroup数据
    scon_ids_to_delete = (
        itermediate_consumergroup_df.alias("srcg")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srcg.srcg_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srcg.SRCG_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_consumergroup_to_delete = (
        master_consumergroup_df.alias("scgr")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scgr.scgr_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scgr.scgr_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scgr.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_consumergroup_delete_count = 0
    if not master_consumergroup_to_delete.isEmpty():
        master_consumergroup_delta_table = DeltaTable.forName(spark, master_consumergroup_table_name)
        master_consumergroup_merge_result = (
            master_consumergroup_delta_table.alias("target")
            .merge(
                master_consumergroup_to_delete.alias("source"),
                """
                target.scgr_id = source.scgr_id AND
                target.scgr_mrkt_code = source.scgr_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_consumergroup_delete_count = max(master_consumergroup_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_consumergroup_insert_count > 0:
        append_table(master_consumergroup_to_insert, master_consumergroup_table_name)
    
    print(f"master consumergroup inserted count: {master_consumergroup_insert_count}, deleted count: {master_consumergroup_delete_count}")

In [0]:
def calc_consumer_hairtype_master(task_id):
    """
    计算并更新consumer hairtype主数据
    全量替换模式：删除特定scon_id的所有hairtype记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_hairtype_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_hair_type"
    
    # 1. 初始化数据源
    itermediate_hairtype_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hair_type") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_hairtype_df = spark.table(master_hairtype_table_name)
    
    # 2. 查询要插入的hairtype数据（步骤提前）
    master_hairtype_to_insert = (
        itermediate_hairtype_df.alias("srht")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srht.srht_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srht.srht_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srht.srht_mrkt_code"),
            F.col("srht.srht_hairtype"),
            F.col("srht.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scht_id"),
            F.col("scon_id").alias("scht_scon_id"),
            F.col("srht_mrkt_code").alias("scht_mrkt_code"),
            F.col("srht_hairtype").alias("scht_hairtype"),
            F.current_timestamp().alias("scht_creation_dt"),
            F.lit("ELC").alias("scht_creation_uid"),
            F.current_timestamp().alias("scht_update_dt"),
            F.lit("ELC").alias("scht_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_hairtype_to_insert = master_hairtype_to_insert.checkpoint(eager=True)
    master_hairtype_insert_count = master_hairtype_to_insert.count()
    
    # 3. 查询要删除的hairtype数据
    scon_ids_to_delete = (
        itermediate_hairtype_df.alias("srht")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srht.srht_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srht.SRHT_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_hairtype_to_delete = (
        master_hairtype_df.alias("scht")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scht.scht_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scht.scht_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scht.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_hairtype_delete_count = 0
    if not master_hairtype_to_delete.isEmpty():
        master_hairtype_delta_table = DeltaTable.forName(spark, master_hairtype_table_name)
        master_hairtype_merge_result = (
            master_hairtype_delta_table.alias("target")
            .merge(
                master_hairtype_to_delete.alias("source"),
                """
                target.scht_id = source.scht_id AND
                target.scht_mrkt_code = source.scht_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_hairtype_delete_count = max(master_hairtype_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_hairtype_insert_count > 0:
        append_table(master_hairtype_to_insert, master_hairtype_table_name)
    
    print(f"master hairtype inserted count: {master_hairtype_insert_count}, deleted count: {master_hairtype_delete_count}")

In [0]:
def calc_consumer_hobby_master(task_id):
    """
    计算并更新consumer hobby主数据
    全量替换模式：删除特定scon_id的所有hobby记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_hobby_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_hobby"
    
    # 1. 初始化数据源
    itermediate_hobby_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_hobby") \
        .where(f"TASK_ID = '{task_id}'") 
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_hobby_df = spark.table(master_hobby_table_name)
    
    # 2. 查询要插入的hobby数据（步骤提前）
    master_hobby_to_insert = (
        itermediate_hobby_df.alias("srhb")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srhb.srhb_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srhb.srhb_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srhb.srhb_mrkt_code"),
            F.col("srhb.srhb_hbby_desc"),
            F.col("srhb.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scho_id"),
            F.col("scon_id").alias("scho_scon_id"),
            F.col("srhb_mrkt_code").alias("scho_mrkt_code"),
            F.col("srhb_hbby_desc").alias("scho_hbby_desc"),
            F.current_timestamp().alias("scho_creation_dt"),
            F.lit("ELC").alias("scho_creation_uid"),
            F.current_timestamp().alias("scho_update_dt"),
            F.lit("ELC").alias("scho_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_hobby_to_insert = master_hobby_to_insert.checkpoint(eager=True)
    master_hobby_insert_count = master_hobby_to_insert.count()
    
    # 3. 查询要删除的hobby数据
    scon_ids_to_delete = (
        itermediate_hobby_df.alias("srhb")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_srcc_id") == F.col("srhb.SRHB_SRCC_ID")) &
            (F.col("sc.scon_mrkt_code") == F.col("srhb.SRHB_MRKT_CODE")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    master_hobby_to_delete = (
        master_hobby_df.alias("scho")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scho.scho_mrkt_code") == F.col("del_ids.scon_mrkt_code")) &
            (F.col("scho.scho_scon_id") == F.col("del_ids.scon_id")),
            "inner"
        )
        .select(F.col("scho.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_hobby_delete_count = 0
    if not master_hobby_to_delete.isEmpty():
        master_hobby_delta_table = DeltaTable.forName(spark, master_hobby_table_name)
        master_hobby_merge_result = (
            master_hobby_delta_table.alias("target")
            .merge(
                master_hobby_to_delete.alias("source"),
                """
                target.scho_id = source.scho_id AND
                target.scho_mrkt_code = source.scho_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_hobby_delete_count = max(master_hobby_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_hobby_insert_count > 0:
        append_table(master_hobby_to_insert, master_hobby_table_name)
    
    print(f"master hobby inserted count: {master_hobby_insert_count}, deleted count: {master_hobby_delete_count}")

In [0]:
def calc_consumer_notes_master(task_id):
    """
    计算并更新consumer notes主数据
    全量替换模式：删除特定scon_id的所有notes记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_notes_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_notes"
    
    # 1. 初始化数据源
    itermediate_notes_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_notes") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_notes_df = spark.table(master_notes_table_name)
    
    # 2. 查询要插入的notes数据（步骤提前）
    master_notes_to_insert = (
        itermediate_notes_df.alias("srno")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srno.srno_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srno.srno_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srno.srno_mrkt_code"),
            F.col("srno.SRNO_SEQ_NUM"),
            F.col("srno.SRNO_TYPE_CODE"),
            F.col("srno.SRNO_LOCATION"),
            F.col("srno.SRNO_NOTE"),
            F.col("srno.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scno_id"),
            F.col("scon_id").alias("scno_scon_id"),
            F.col("srno_mrkt_code").alias("scno_mrkt_code"),
            F.col("SRNO_SEQ_NUM").alias("scno_seq_num"),
            F.col("SRNO_TYPE_CODE").alias("scno_type_code"),
            F.col("SRNO_LOCATION").alias("scno_location"),
            F.col("SRNO_NOTE").alias("scno_note"),
            F.current_timestamp().alias("scno_creation_dt"),
            F.lit("ELC").alias("scno_creation_uid"),
            F.current_timestamp().alias("scno_update_dt"),
            F.lit("ELC").alias("scno_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_notes_to_insert = master_notes_to_insert.checkpoint(eager=True)
    master_notes_insert_count = master_notes_to_insert.count()
    
    # 3. 查询要删除的notes数据
    scon_ids_to_delete = (
        itermediate_notes_df.alias("srno")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srno.srno_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srno.SRNO_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    master_notes_to_delete = (
        master_notes_df.alias("scno")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scno.scno_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scno.scno_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scno.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_notes_delete_count = 0
    if not master_notes_to_delete.isEmpty():
        master_notes_delta_table = DeltaTable.forName(spark, master_notes_table_name)
        master_notes_merge_result = (
            master_notes_delta_table.alias("target")
            .merge(
                master_notes_to_delete.alias("source"),
                """
                target.scno_id = source.scno_id AND
                target.scno_mrkt_code = source.scno_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_notes_delete_count = max(master_notes_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_notes_insert_count > 0:
        append_table(master_notes_to_insert, master_notes_table_name)
    
    print(f"master notes inserted count: {master_notes_insert_count}, deleted count: {master_notes_delete_count}")

In [0]:
def calc_consumer_remark_master(task_id):
    """
    计算并更新consumer remark主数据
    全量替换模式：删除特定scon_id的所有remark记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_remark_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_remark"
    
    # 1. 初始化数据源
    itermediate_remark_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_remark") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_remark_df = spark.table(master_remark_table_name)
    
    # 2. 查询要插入的remark数据（步骤提前）
    master_remark_to_insert = (
        itermediate_remark_df.alias("srcr")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srcr.srcr_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srcr.srcr_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srcr.srcr_mrkt_code"),
            F.col("srcr.SRCR_RMAK_DT"),
            F.col("srcr.SRCR_RMAK_CODE"),
            F.col("srcr.SRCR_RMAK_DESCRIPTION"),
            F.col("srcr.batch_id")
        )
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scre_id"),
            F.col("scon_id").alias("scre_scon_id"),
            F.col("srcr_mrkt_code").alias("scre_mrkt_code"),
            F.col("SRCR_RMAK_DT").alias("scre_rmak_dt"),
            F.col("SRCR_RMAK_CODE").alias("scre_rmak_code"),
            F.col("SRCR_RMAK_DESCRIPTION").alias("scre_rmak_description"),
            F.current_timestamp().alias("scre_creation_dt"),
            F.lit("ELC").alias("scre_creation_uid"),
            F.current_timestamp().alias("scre_update_dt"),
            F.lit("ELC").alias("scre_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_remark_to_insert = master_remark_to_insert.checkpoint(eager=True)
    master_remark_insert_count = master_remark_to_insert.count()
    
    # 3. 查询要删除的remark数据
    scon_ids_to_delete = (
        itermediate_remark_df.alias("srcr")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srcr.srcr_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srcr.SRCR_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    master_remark_to_delete = (
        master_remark_df.alias("scre")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scre.scre_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scre.scre_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scre.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_remark_delete_count = 0
    if not master_remark_to_delete.isEmpty():
        master_remark_delta_table = DeltaTable.forName(spark, master_remark_table_name)
        master_remark_merge_result = (
            master_remark_delta_table.alias("target")
            .merge(
                master_remark_to_delete.alias("source"),
                """
                target.scre_id = source.scre_id AND
                target.scre_mrkt_code = source.scre_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_remark_delete_count = max(master_remark_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_remark_insert_count > 0:
        append_table(master_remark_to_insert, master_remark_table_name)
    
    print(f"master remark inserted count: {master_remark_insert_count}, deleted count: {master_remark_delete_count}")

In [0]:
def calc_consumer_terms_master(task_id):
    """
    计算并更新consumer terms主数据
    全量替换模式：删除特定scon_id的所有terms记录后重新插入
    
    Args:
        task_id: 任务ID
    """
    master_terms_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_terms"
    
    # 1. 初始化数据源
    itermediate_terms_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_terms") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_terms_df = spark.table(master_terms_table_name)
    
    # 2. 查询要插入的terms数据（步骤提前）
    master_terms_to_insert = (
        itermediate_terms_df.alias("srct")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srct.srct_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srct.srct_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srct.srct_mrkt_code"),
            F.col("srct.SRCT_TERMS_CODE"),
            F.col("srct.SRCT_TERMS_DESCRIPTION"),
            F.col("srct.SRCT_TERMS_ACCEPT_DT"),
            F.col("srct.SRCT_TERMS_VERSION"),
            F.col("srct.batch_id")
        )
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scte_id"),
            F.col("scon_id").alias("scte_scon_id"),
            F.col("srct_mrkt_code").alias("scte_mrkt_code"),
            F.col("SRCT_TERMS_CODE").alias("scte_terms_code"),
            F.col("SRCT_TERMS_DESCRIPTION").alias("scte_terms_description"),
            F.col("SRCT_TERMS_ACCEPT_DT").alias("scte_terms_accept_dt"),
            F.col("SRCT_TERMS_VERSION").alias("scte_terms_version"),
            F.current_timestamp().alias("scte_creation_dt"),
            F.lit("ELC").alias("scte_creation_uid"),
            F.current_timestamp().alias("scte_update_dt"),
            F.lit("ELC").alias("scte_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_terms_to_insert = master_terms_to_insert.checkpoint(eager=True)
    master_terms_insert_count = master_terms_to_insert.count()
    
    # 3. 查询要删除的terms数据
    scon_ids_to_delete = (
        itermediate_terms_df.alias("srct")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srct.srct_mrkt_code")) &
            (F.col("sc.scon_srcc_id") == F.col("srct.SRCT_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id").alias("scon_id"), F.col("sc.scon_mrkt_code").alias("scon_mrkt_code"))
        .distinct()
    )
    
    master_terms_to_delete = (
        master_terms_df.alias("scte")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scte.scte_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scte.scte_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scte.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_terms_delete_count = 0
    if not master_terms_to_delete.isEmpty():
        master_terms_delta_table = DeltaTable.forName(spark, master_terms_table_name)
        master_terms_merge_result = (
            master_terms_delta_table.alias("target")
            .merge(
                master_terms_to_delete.alias("source"),
                """
                target.scte_id = source.scte_id AND
                target.scte_mrkt_code = source.scte_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_terms_delete_count =max(master_terms_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_terms_insert_count > 0:
        append_table(master_terms_to_insert, master_terms_table_name)
    
    print(f"master terms inserted count: {master_terms_insert_count}, deleted count: {master_terms_delete_count}")
    

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("5.4_generate_master_other_tables", "05-4", "consumerlist", task_id=task_id) as logger:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")

    calc_consumer_crossbrand_optin_master(task_id)
    calc_consumer_auxiliaryattribute_master(task_id)
    calc_consumer_hairconcerns_master(task_id)
    calc_consumer_makeupconcerns_master(task_id)
    calc_consumer_skinconcerns_master(task_id)
    calc_consumer_customattr_master(task_id)
    calc_consumer_group_master(task_id)
    calc_consumer_hairtype_master(task_id)
    calc_consumer_hobby_master(task_id)
    calc_consumer_notes_master(task_id)
    calc_consumer_remark_master(task_id)
    calc_consumer_terms_master(task_id)